## Process output of simulations

In [1]:
import os
import glob
import gzip
import math
import random
import pickle

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.colors import LogNorm
import shapely.wkt as wkt
from shapely.geometry import Point, LineString, box
from shapely.ops import nearest_points
import lxml.etree as ET
import tqdm
import wandb
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, Subset
import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.transforms import LineGraph
import processing_io as pio
import re 

districts = gpd.read_file("../../../../data/visualisation/districts_paris.geojson")

# Parameters to adapt
districts_of_policy_implementation = [5, 6, 7]
is_for_1pm = False
plot_in_percentage = True

string_is_for_1pm = "pop_1pm" if is_for_1pm else "pop_1pct"
string_district_of_interest = "_".join([str(d) for d in districts_of_policy_implementation])

path = "../../../../data/" +  string_is_for_1pm + "_simulations/"
basecase_subdir = pio.get_subdirs(path + string_is_for_1pm + "_basecase/")
comparison_subdir = pio.get_subdirs(path + string_is_for_1pm + "_policy_in_zone_2")

result_path_basecase_mean = "results/" + string_is_for_1pm + "_basecase_average_output_links.geojson"
result_path_comparison_mean = "results/gdf_" + string_is_for_1pm + "_policy_in_" + string_district_of_interest + ".geojson"
result_path_difference = "results/gdf_" + string_is_for_1pm + "_difference.geojson"

compute_comparison_with_basecase = False

# THis is old version

In [2]:
random_seed_2_df_basecase_output_links = pio.create_dic_seed_2_output_links(subdir=basecase_subdir)
random_seed_2_df_basecase_trips = pio.create_dic_seed_2_eqasim_trips_given_output_trips(subdir=basecase_subdir)
basecase_output_links_gdfs = list(random_seed_2_df_basecase_output_links.values())
gdf_basecase_mean = pio.compute_average_or_median_geodataframe(geodataframes=basecase_output_links_gdfs, column_name="vol_car", is_mean=True)
gdf_basecase_mean = gdf_basecase_mean.rename(columns={"osm:way:highway": "highway"})
gdf_basecase_mean.to_file(result_path_basecase_mean, driver='GeoJSON')

In [3]:
random_seed_2_df_basecase_trips['{1}']

,person,trip_number,trip_id,dep_time,trav_time,wait_time,traveled_distance,euclidean_distance,main_mode,longest_distance_mode,...,start_facility_id,start_link,start_x,start_y,end_facility_id,end_link,end_x,end_y,first_pt_boarding_stop,last_pt_egress_stop
0,1,1,1_1,09:41:29,00:12:46,00:00:00,920,707,walk,walk,...,home_1,262151,649869.120000,6.860582e+06,sec_409991,515043,649443.100000,6.860017e+06,NaN,NaN
1,1,2,1_2,11:11:29,00:14:24,00:00:00,1370,1038,pt,walk,...,sec_409991,515043,649443.100000,6.860017e+06,sec_229585,537320,649222.230000,6.861031e+06,NaN,NaN
2,10,1,10_1,10:26:16,00:08:21,00:00:00,602,463,walk,walk,...,home_7,43330,656662.136372,6.860692e+06,outside_1,482724,657116.961349,6.860780e+06,NaN,NaN
3,10,2,10_2,15:56:16,00:00:01,00:00:00,1802,1801,outside,outside,...,outside_1,482724,657116.961349,6.860780e+06,outside_2,112845,658918.022379,6.860728e+06,NaN,NaN
4,10,3,10_3,16:01:41,00:02:01,00:00:00,146,111,walk,walk,...,outside_2,112845,658918.022379,6.860728e+06,outside_3,118289,658968.557163,6.860628e+06,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142753,9997623,2,9997623_2,11:20:19,00:05:11,00:00:00,2147,1518,car,car,...,work_2343689,561817,652993.810000,6.859080e+06,outside_26,108473,653508.573590,6.857651e+06,NaN,NaN
142754,9998180,1,9998180_1,07:13:33,00:07:09,00:00:00,3794,2547,car_passenger,car_passenger,...,outside_116,108472,653521.987413,6.857658e+06,edu_11045,581070,651724.100000,6.859464e+06,NaN,NaN
142755,9998180,2,9998180_2,16:45:23,00:28:42,00:00:00,3540,2630,pt,pt,...,edu_11045,581070,651724.100000,6.859464e+06,outside_57,339663,654302.838874,6.858944e+06,NaN,NaN
142756,9998181,1,9998181_1,10:50:30,00:29:14,00:00:00,6152,5457,pt,pt,...,outside_57,339663,654302.838874,6.858944e+06,work_2351574,260151,650788.900000,6.863119e+06,NaN,NaN


In [4]:
# Helper function to convert time strings to total seconds
def convert_time_to_seconds(df, column_name):
    df[column_name] = pd.to_timedelta(df[column_name]).dt.total_seconds()
    return df

# Calculate average travel time and routed distance per mode across all seeds
# def calculate_avg_mode_stats(single_mode_stats_list: list):
#     mode_stats_list = []

#     for df in single_mode_stats_list:
#         # Convert 'trav_time' to total seconds for aggregation
#         df = convert_time_to_seconds(df, 'trav_time')

#         # Aggregate travel time and traveled distance by main mode
#         mode_stats = df.groupby('main_mode').agg({
#             'trav_time': 'sum',
#             'traveled_distance': 'sum'
#         }).reset_index()
#         mode_stats_list.append(mode_stats)

#     # Concatenate all mode_stats dataframes
#     all_mode_stats = pd.concat(mode_stats_list, ignore_index=True)

#     # Calculate the average across all seeds
#     average_mode_stats = all_mode_stats.groupby('main_mode').agg({
#         'trav_time': 'mean',
#         'traveled_distance': 'mean'
#     }).reset_index()

#     # Rename columns for clarity
#     average_mode_stats.columns = ['main_mode', 'avg_trav_time_seconds', 'avg_traveled_distance']
    
#     return average_mode_stats

# Calculate average travel time, routed distance, and trip count per mode across all seeds
def calculate_avg_mode_stats(single_mode_stats_list: list):
    mode_stats_list = []

    for df in single_mode_stats_list:
        # Convert 'trav_time' to total seconds for aggregation
        df = convert_time_to_seconds(df, 'trav_time')

        # Aggregate travel time, traveled distance, and trip count by main mode
        mode_stats = df.groupby('main_mode').agg({
            'trav_time': 'sum',
            'traveled_distance': 'sum',
            'trip_id': 'count'  # Count trips by counting unique trip_ids
        }).reset_index()
        mode_stats_list.append(mode_stats)

    # Concatenate all mode_stats dataframes
    all_mode_stats = pd.concat(mode_stats_list, ignore_index=True)

    # Calculate the average across all seeds
    average_mode_stats = all_mode_stats.groupby('main_mode').agg({
        'trav_time': 'mean',
        'traveled_distance': 'mean',
        'trip_id': 'mean'  # Calculate average trip count
    }).reset_index()

    # Rename columns for clarity
    average_mode_stats.columns = ['main_mode', 'avg_trav_time_seconds', 'avg_traveled_distance', 'average_trip_count']
    
    return average_mode_stats

df_average_mode_stats = calculate_avg_mode_stats(random_seed_2_df_basecase_trips.values())
df_average_mode_stats.to_csv("results/pop_1pct_basecase_average_mode_stats.csv", index=False)

In [5]:
df_average_mode_stats

,main_mode,avg_trav_time_seconds,avg_traveled_distance,average_trip_count
0,bike,3.203165e+06,9.934483e+06,2980.214286
1,car,1.644894e+07,1.665059e+08,35050.166667
2,car_passenger,3.593220e+06,3.807044e+07,8434.000000
3,outside,1.851514e+04,2.495273e+07,25077.976190
4,pt,6.445410e+07,2.208262e+08,39754.333333
5,walk,2.539061e+07,3.048681e+07,31460.571429


In [6]:
if compute_comparison_with_basecase:
    random_seed_2_df_comparison = pio.create_dic_seed_2_output_links(subdir = comparison_subdir)
    geodataframes_comparison = list(random_seed_2_df_comparison.values())
    gdf_comparison_mean = pio.compute_average_or_median_geodataframe(geodataframes=geodataframes_comparison, column_name="vol_car", is_mean=True)
    gdf_comparison_mean_extended = pio.extend_geodataframe(gdf_base = gdf_basecase_mean, gdf_to_extend=gdf_comparison_mean, column_to_extend='highway', new_column_name='highway')
    gdf_basecase_without_unnecessary_columns = pio.remove_columns(gdf_with_correct_columns=gdf_comparison_mean_extended, gdf_to_be_adapted=gdf_basecase_mean)
    gdf_basecase_difference = pio.compute_difference_geodataframe(gdf_to_substract_from=gdf_comparison_mean_extended, gdf_to_substract=gdf_basecase_without_unnecessary_columns, column_name= 'vol_car')
    gdf_comparison_mean_extended.to_file(result_path_comparison_mean, driver='GeoJSON')
    gdf_basecase_difference.to_file(result_path_difference, driver='GeoJSON')